# Nitshio — QLoRA en Kaggle/Colab

Este notebook entrena **solo en la nube**. La T1200 local de 4 GB no entrena; localmente se sigue usando Ollama para inferencia.

## Antes de ejecutar

1. Sube el proyecto `Nitshio` a `/kaggle/working/Nitshio` (o define `NITSHIO_DIR`).
2. Sube el dataset JSONL validado (`training/datasets/synthetic.jsonl` o el que uses).
3. Activa una GPU en el entorno (T4/P100 o superior).
4. Ejecuta las celdas en orden. La exportación GGUF requiere clonar y compilar `llama.cpp`.

> El modelo base, los parámetros QLoRA y la cuantización se leen de `config/config.yaml`. No cambies nombres de modelo en el código.

In [ ]:
import os, subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
PROJECT_DIR = os.environ.get('NITSHIO_DIR', '/kaggle/working/Nitshio')
print('PROJECT_DIR =', PROJECT_DIR)
assert os.path.isdir(PROJECT_DIR), 'Sube el proyecto o define NITSHIO_DIR'

In [ ]:
os.chdir(PROJECT_DIR)
print('Directorio actual:', os.getcwd())
%pip install -q -r requirements-train.txt
# Si Unsloth pide una instalación específica para tu GPU, sigue su salida:
# https://docs.unsloth.ai/get-started/installing-+-updating
print('Dependencias instaladas. Si hubo actualización de torch, reinicia el kernel antes de entrenar.')

In [ ]:
import json
from pathlib import Path

config = Path('config/config.yaml')
assert config.exists(), 'Falta config/config.yaml en el proyecto subido'
print(config.read_text(encoding='utf-8')[:1200])

synthetic = Path('training/datasets/synthetic.jsonl')
seed = Path('training/datasets/seed.jsonl')
dataset = synthetic if synthetic.exists() else seed
print('Dataset elegido:', dataset)
assert dataset.exists(), 'Sube un dataset JSONL'

In [ ]:
subprocess.run(
    [sys.executable, 'training/validate_dataset.py', str(dataset), '--strict', '--split-dir', 'training/datasets/splits'],
    check=True,
)
print('Dataset validado y particionado.')

In [ ]:
OUTPUT_LORA = Path('/kaggle/working/nitshio-lora')
subprocess.run(
    [sys.executable, 'training/finetune_qlora.py',
     '--dataset', str(dataset),
     '--output-dir', str(OUTPUT_LORA)],
    check=True,
)
print('Adaptadores en', OUTPUT_LORA)

In [ ]:
# Clonar y compilar llama.cpp para convertir a GGUF y cuantizar.
LLAMA_DIR = Path('/kaggle/working/llama.cpp')
if not LLAMA_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggerganov/llama.cpp', str(LLAMA_DIR)], check=True)
subprocess.run(['cmake', '-S', str(LLAMA_DIR), '-B', str(LLAMA_DIR / 'build'), '-DGGML_CUDA=ON'], check=True)
subprocess.run(['cmake', '--build', str(LLAMA_DIR / 'build'), '--config', 'Release', '-j', '--target', 'llama-quantize'], check=True)
print('llama.cpp listo en', LLAMA_DIR)

In [ ]:
OUTPUT_GGUF = Path('/kaggle/working/nitshio-gguf')
subprocess.run(
    [sys.executable, 'training/export_gguf.py',
     '--adapter-dir', str(OUTPUT_LORA),
     '--llama-cpp-dir', str(LLAMA_DIR),
     '--output-dir', str(OUTPUT_GGUF)],
    check=True,
)
print('GGUF en', OUTPUT_GGUF)

In [ ]:
import shutil

zip_base = '/kaggle/working/nitshio-entrega'
shutil.make_archive(zip_base, 'zip', '/kaggle/working', 'nitshio-gguf')
print('Descarga:', zip_base + '.zip')
print('En local:')
print('  ollama create nitshio-v2 -f models/Modelfile')
print('  python training/evaluate.py --models nitshio-v1 nitshio-v2 --record-metrics')